# EcoRewind 2.0 — Phase 0: Earth Engine re-export (Colab, CPU runtime)

Earth Engine computes the composites on Google's servers and writes Cloud-Optimised GeoTIFFs **straight into your Google Drive** (`MyDrive/EcoRewind2_exports/`). Nothing is downloaded to your laptop, and this notebook uses no GPU.

Order: setup → authenticate → **preflight (go/stop)** → dry run → 1 test file per kind → everything → monitor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- fill these in -----------------------------------------------------------
PROJECT = 'your-ee-project-id'          # Earth Engine cloud project (code.earthengine.google.com -> project)
import glob, os
ZIP = max(glob.glob('/content/drive/MyDrive/EcoRewind2/ecorewind*.zip'), key=os.path.getmtime)  # newest code zip
print('code:', ZIP)
REPORTS = '/content/drive/MyDrive/EcoRewind2/reports'   # preflight, manifests, audit persist here
# ----------------------------------------------------------------------------
import os
os.environ['ECOREWIND_REPORTS'] = REPORTS
os.makedirs(REPORTS, exist_ok=True)
!rm -rf /content/ecorewind && unzip -q "$ZIP" -d /content
!pip -q install earthengine-api pyproj rasterio==1.4.3
%cd /content/ecorewind
!python -m pytest -q tests

### Authenticate Earth Engine (once per session)

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
print('EE ok:', ee.String('hello').getInfo())

### 1. Preflight: catalogue check, scene counts per quarter, S2 product choice, SAR sanity check
Read the verdict. **If it says STOP, don't export**; send me `gee_preflight.md`.

In [ ]:
!python gee/preflight.py --project $PROJECT
from IPython.display import Markdown
Markdown(open(f'{REPORTS}/gee_preflight.md').read())

### 2. Dry run: list every file that will be exported (no tasks submitted)

In [ ]:
for s in ['export_trackA_s2', 'export_trackB_hls', 'export_s1', 'export_static']:
    !python gee/{s}.py --dry-run | tail -3

### 3. Test: one file of each kind
Wait until these 4 finish (cell 5 shows progress), open one in Drive, and run the audit notebook on them with `--max-files` if you want. Then continue.

In [ ]:
!python gee/export_static.py    --project $PROJECT --limit 1
!python gee/export_trackA_s2.py  --project $PROJECT --limit 1 --site mississippi --only 2021Q4
!python gee/export_trackB_hls.py --project $PROJECT --limit 1 --site everglades  --only 2016Q4
!python gee/export_s1.py         --project $PROJECT --limit 1 --site mississippi --only 2019Q2

### 4. Submit everything
Already-submitted files are skipped, so re-running this cell is safe (e.g. after a disconnect). 320 tasks; Earth Engine runs a few at a time, so allow several hours. Colab can be closed once tasks are submitted: they run on Earth Engine, not in Colab.

In [ ]:
!python gee/export_static.py    --project $PROJECT
!python gee/export_trackA_s2.py  --project $PROJECT
!python gee/export_trackB_hls.py --project $PROJECT
!python gee/export_s1.py         --project $PROJECT

### 5. Monitor
Also visible at code.earthengine.google.com → Tasks. Failed tasks: fix the cause, then rerun the matching cell-4 line with `--force --only <period>`.

In [ ]:
!python gee/monitor.py --project $PROJECT